In [6]:

import numpy as np
from pathlib import Path

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data = np.load(project_dir / "results" / "raw_window_flattened_split.npz")

X_train_xgb = data["X_train"]
X_test_xgb = data["X_test"]
y_train_xgb = data["y_train"]
y_test_xgb = data["y_test"]




### Train XGBoost Regressor


In [7]:

from xgboost import XGBRegressor

# Initialize XGBoost
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.9,
    reg_lambda=5,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

# Train the model
xgb_model.fit(X_train_xgb, y_train_xgb)

print("XGBoost training completed!")
print("Number of trees:", xgb_model.n_estimators)
print("Training samples:", X_train_xgb.shape[0])
print("Input features:", X_train_xgb.shape[1])


XGBoost training completed!
Number of trees: 300
Training samples: 500
Input features: 1800



### Evaluate XGBoost Regression


In [8]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predict fatigue scores
y_pred_xgb = xgb_model.predict(X_test_xgb)

# Keep predictions within the 0–10 fatigue scale
y_pred_xgb = np.clip(y_pred_xgb, 0, 10)

# Calculate evaluation metrics
xgb_mae = mean_absolute_error(y_test_xgb, y_pred_xgb)
xgb_mse = mean_squared_error(y_test_xgb, y_pred_xgb)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_test_xgb, y_pred_xgb)

print(f"MAE: {xgb_mae:.4f}")
print(f"MSE: {xgb_mse:.4f}")
print(f"RMSE: {xgb_rmse:.4f}")
print(f"R² Score: {xgb_r2:.4f}")


MAE: 1.9835
MSE: 5.1353
RMSE: 2.2661
R² Score: 0.1322



### XGBoost Classification Accuracy


In [9]:

from sklearn.metrics import accuracy_score

# Convert fatigue scores into Low, Moderate, High
y_test_xgb_levels = np.digitize(y_test_xgb, bins=[4, 7])
y_pred_xgb_levels = np.digitize(y_pred_xgb, bins=[4, 7])

# Calculate classification accuracy
xgb_accuracy = accuracy_score(
    y_test_xgb_levels,
    y_pred_xgb_levels
)

print(f"XGBoost Accuracy: {xgb_accuracy * 100:.2f}%")


XGBoost Accuracy: 55.91%



### Evaluate XGBoost High Fatigue Recall


In [10]:

# Identify actual High Fatigue windows
high_mask_xgb = y_test_xgb_levels == 2

# Count actual and correctly detected cases
total_high_xgb = high_mask_xgb.sum()

detected_high_xgb = (
    y_pred_xgb_levels[high_mask_xgb] == 2
).sum()

# Calculate High Fatigue Recall
xgb_high_recall = (
    detected_high_xgb / total_high_xgb
    if total_high_xgb > 0 else 0
)

print("Actual High Fatigue Windows:", total_high_xgb)
print("Correctly Detected:", detected_high_xgb)
print(f"High Fatigue Recall: {xgb_high_recall * 100:.2f}%")


Actual High Fatigue Windows: 19
Correctly Detected: 0
High Fatigue Recall: 0.00%



### Save XGBoost 3-Minute Window Flattened Model


In [11]:

# Create model directory
model_dir = project_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)

# Save the trained flattened XGBoost model
model_path = model_dir / "xgboost_3min_flattened.json"

xgb_model.save_model(model_path)

print("XGBoost model saved successfully!")
print("Location:", model_path)


XGBoost model saved successfully!
Location: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\models\xgboost_3min_flattened.json



## XGBoost — 3-Minute Window Flattened Model

### Model Configuration

- **Window Duration:** 3 minutes (180 seconds)
- **Readings per Window:** 360
- **Sensor Signals:** 5
- **Input Features:** 1,800 (Flattened Raw Readings)
- **Training Windows:** 500
- **Testing Windows:** 127
- **Number of Trees:** 300
- **Maximum Tree Depth:** 3
- **Learning Rate:** 0.05
- **Minimum Child Weight:** 5

### Model Evaluation Results

- **Classification Accuracy:** 55.91%
- **MAE (Mean Absolute Error):** 1.9835
- **MSE (Mean Squared Error):** 5.1353
- **RMSE (Root Mean Squared Error):** 2.2661
- **R² Score:** 0.1322
- **Actual High Fatigue Windows:** 19
- **Correctly Detected High Fatigue Windows:** 0
- **High Fatigue Recall:** 0.00%
